## Iteration Changelog
- Added PCA and feature-reduction workflow on cleaned model-ready data.
- Added variance/correlation review and dimensionality reduction notes.
- Open issues: compare PCA components by tournament period for stability.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = ROOT / "data" / "cleanedGambling.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)

## PCA & Feature Reduction

### Workflow
1) choose numeric model-ready features,
2) remove near-constant signals,
3) inspect correlation redundancy,
4) run PCA and analyze cumulative variance.

In [ ]:
numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
exclude = {"won_flag", "lost_flag", "balance"}
feature_cols = [c for c in numeric_cols if c not in exclude]

X = df[feature_cols].copy().fillna(0.0)
var = X.var(ddof=0)
low_var = var[var < 1e-5].index.tolist()
X = X.drop(columns=low_var)

corr = X.corr(numeric_only=True).abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if (upper[c] > 0.98).any()]
X_reduced = X.drop(columns=to_drop)

print(f"Start cols: {len(feature_cols)}")
print(f"Low variance removed: {len(low_var)}")
print(f"Highly correlated removed: {len(to_drop)}")
print(f"Final PCA input cols: {X_reduced.shape[1]}")

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reduced)

pca = PCA(n_components=min(25, X_scaled.shape[1]))
pcs = pca.fit_transform(X_scaled)

ev = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))],
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_variance": np.cumsum(pca.explained_variance_ratio_),
})
ev.head(15)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(ev)+1), ev["cumulative_variance"], marker="o")
plt.axhline(0.90, color="red", linestyle="--", linewidth=1)
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Cumulative Variance")
plt.grid(alpha=0.25)
plt.show()

proj = pd.DataFrame(pcs[:, :2], columns=["PC1", "PC2"])
proj["won_flag"] = df["won_flag"].values if "won_flag" in df.columns else 0

plt.figure(figsize=(7, 5))
for val, grp in proj.groupby("won_flag"):
    plt.scatter(grp["PC1"], grp["PC2"], s=8, alpha=0.35, label=f"won_flag={val}")
plt.title("PCA Projection (PC1 vs PC2)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()